#### This notebook is used to experiment with DTGraph rules and transformations.

In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

In [3]:
hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [ ]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

### Old Node Rules

In [5]:
rule1 = Rule(
    """
MATCH (p:Person)
GENERATE
(x = (p):Test { value = "A" })
"""
)

rule2 = Rule(
    """
MATCH (p:Person)
GENERATE
(x = (p):Test { value = "B" })
"""
)

explosion = Rule(
    """
MATCH (a:Person), (b:Person)
GENERATE
(x = (a):Person)-[():CONNECTED]->(y = (b):Person)
"""
)

type_mutation = Rule(
    """
MATCH (p:Person)
GENERATE
(x = (p):Movie)
"""
)

control = Rule(
    """
MATCH (p:Person)
GENERATE
(x = (p):)
"""
)

### Testing PG-Schema

#### Node Rules Testing

In [ ]:
rule_valid_actor = Rule(
    """
MATCH (p:Person)
=>
(x = (p):Actor {
    name = p.name
})
"""
)

rule_valid_movie = Rule(
    """
MATCH (m:Movie)
=>
(x = (m):Movie {
    title = m.title
});
(x = (m):Film,Season {
    title = 1234
});
(x = (p):)-[():ACTED_IN ]->(y = (m):)
"""
)

rule_extra_property = Rule(
    """
MATCH (p:Person)
GENERATE
(x = (p):Actor {
    name = p.name,
    age = "30"
})
"""
)

# DTGraph doesnt allow this syntax of having extra label
# rule_extra_label = Rule('''
# MATCH (p:Person)
# GENERATE
# (x = (p):Actor:VIP {
#     name = p.name
# })                    
# ''')

# DTGraph doesnt allow this syntax of having no label
# rule_no_label = Rule('''
# MATCH (p:Person)
# GENERATE
# (x = (p) {
#     name = p.name
# })
# ''')

node_rules = [
    {
        "rule": rule_valid_actor,
        "text": """
MATCH (p:Person)
GENERATE
(x = (p):Actor:VIP {
    name = p.name
})
""",
    }
]

#### Edge Rules Testing

In [ ]:
rule_valid_edge = Rule(
    """
MATCH (p:Actor), (m:Movie)
GENERATE
(x = (p):)-[():ACTED_IN ]->(y = (m):)
"""
)


rule_edge_with_year = Rule(
    """
MATCH (p:Actor), (m:Movie)
GENERATE
(x = (p):)-[():ACTED_IN { year = "2023" }]->(y = (m):)
"""
)


edge_rules = [
    {
        "rule": rule_valid_edge,
        "text": """
MATCH (p:Actor), (m:Movie)
GENERATE
(x = (p):)-[():ACTED_IN ]->(y = (m):)
""",
    }
]

#### Testing Schema

In [ ]:
## PG-Schema integration

from pg_schema.loader import SchemaLoader
from pg_schema.precheck import precheck_rule


def reload_all():
    import importlib
    import pg_schema.matcher as matcher
    import pg_schema.validator as validator
    import pg_schema.precheck as precheck

    importlib.reload(matcher)
    importlib.reload(validator)
    importlib.reload(precheck)

    return precheck


def run_tests():
    precheck = reload_all()
    schema = SchemaLoader("../dtgraph/pg_schema/testSchema.json").load()

    all_passed = True

    for item in node_rules:
        rule_str = item["text"]

        try:
            if "GENERATE" in rule_str:
                generate_clause = rule_str.split("GENERATE")[1]
            elif "=>" in rule_str:
                generate_clause = rule_str.split("=>")[1]
            else:
                raise ValueError("No GENERATE clause found")

            precheck.precheck_rule(generate_clause, schema, rule_str)

        except Exception as e:
            all_passed = False

            print("\nRULE FAILED:")
            print(rule_str)

            msg = str(e)
            lines = msg.split("\n")
            filtered = [
                line.strip(" -") for line in lines if line.strip().startswith("-")
            ]

            if filtered:
                print("Reason:", filtered[0])
            else:
                print("Reason:", msg)

    if all_passed:
        print("\nAll rules conform to schema")

In [ ]:
run_tests()

### Applying Transformations

In [ ]:
## Applying transformation rules
my_transform = Transformation([rule1, rule2, explosion, type_mutation, control])
my_transform.apply_on(graph)

In [ ]:
my_transform.abort()